# NutriCoach — Deep Learning Capstone (single-notebook run)

Runs the **entire pipeline in one Colab runtime** so state persists throughout.
It drives the clean `src/nutricoach` modules (cloned from GitHub).

**Phase 1 (tabular):** calorie regression · Nutri-Score classification · Conditional VAE  
**Phase 2 (RAG):** knowledge corpus · domain-fine-tuned retriever · retrieval eval · RAG demo

> Set **Runtime → Change runtime type → GPU** before running.  
> Add a Colab Secret 🔑 `KAGGLE_API_TOKEN` (your Kaggle access token) for the dataset download.  
> Add a Colab Secret 🔑 `HF_TOKEN` (a Hugging Face token with **write** access) to push results.


## 0 · Setup (clone repo, install, Kaggle auth)


In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/shekeraoleksandr/AI-NutriCoach.git'

if IN_COLAB:
    repo = '/content/AI-NutriCoach'
    if not os.path.exists(repo):
        subprocess.run(['git', 'clone', REPO_URL, repo], check=True)
    os.chdir(repo)
    subprocess.run(['git', 'pull'], check=False)
    subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    subprocess.run(['pip', 'install', '-q', 'wikipedia-api'], check=True)
    try:
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    except Exception as e:
        print('!! Add Colab Secret KAGGLE_API_TOKEN (or an access_token file). ', e)

sys.path.insert(0, 'src')
print('cwd:', os.getcwd(), '| Colab:', IN_COLAB)


In [ ]:
from nutricoach import (config as C, data, models_tabular as M, cvae,
                        ingest, retriever, pairs, rag, evaluate as E)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
print('modules loaded · GPU available:', torch.cuda.is_available())


---
# Phase 1 — Tabular nutrition analysis


## 1.1 · Download & clean


In [ ]:
csv_path = data.download_raw()          # Open Food Facts via kagglehub
df = data.load_clean(csv_path)
print('shape:', df.shape)
df.head()


## 1.2 · EDA


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[C.REGRESSION_TARGET], bins=60, ax=ax[0]); ax[0].set_title('kcal / 100g')
order = C.NUTRISCORE_GRADES
df[C.CLASSIFICATION_TARGET].value_counts().reindex(order).plot.bar(ax=ax[1])
ax[1].set_title('Nutri-Score class balance'); plt.tight_layout(); plt.show()


In [ ]:
print('Nutri-Score counts:')
print(df[C.CLASSIFICATION_TARGET].value_counts(dropna=False))
print('\nCorrelation with calories:')
print(df[C.NUTRIENT_FEATURES + [C.REGRESSION_TARGET]].corr()[C.REGRESSION_TARGET].sort_values())


In [ ]:
df.to_parquet(C.PROCESSED_DIR / 'food_clean.parquet', index=False)
print('saved processed frame')


## 1.3 · Task 1 — calorie regression
Compare Random Forest / XGBoost / MLP; keep the best (lowest RMSE).


In [ ]:
Xtr, Xte, ytr, yte = data.split_regression(df)
reg_results = {}
for name, model in M.build_regressors().items():
    model.fit(Xtr, ytr)
    reg_results[name] = E.regression_metrics(yte, model.predict(Xte))
reg_df = pd.DataFrame(reg_results).T; reg_df


In [ ]:
best_name = reg_df['RMSE'].idxmin()
best = M.build_regressors()[best_name]
best.fit(df[C.NUTRIENT_FEATURES].values, df[C.REGRESSION_TARGET].values)
M.save_model(best, M.REG_PATH); print('saved regressor:', best_name)


## 1.4 · Task 2 — Nutri-Score classification
Imbalanced — use `class_weight='balanced'`; report macro-F1, not just accuracy.


In [ ]:
Xtr, Xte, ytr, yte = data.split_classification(df)
# XGBoost needs numeric labels -> encode grades a..e as 0..4 (consistent across models)
g2i = {g: i for i, g in enumerate(C.NUTRISCORE_GRADES)}
ytr_e = np.array([g2i[y] for y in ytr]); yte_e = np.array([g2i[y] for y in yte])
clf_results = {}
for name, model in M.build_classifiers(class_weight='balanced').items():
    model.fit(Xtr, ytr_e)
    clf_results[name] = E.classification_metrics(yte_e, model.predict(Xte))
clf_df = pd.DataFrame(clf_results).T; clf_df


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
best_clf_name = clf_df['macro_f1'].idxmax()
best_clf = M.build_classifiers(class_weight='balanced')[best_clf_name].fit(Xtr, ytr_e)
labels_i = list(range(len(C.NUTRISCORE_GRADES)))
ConfusionMatrixDisplay(confusion_matrix(yte_e, best_clf.predict(Xte), labels=labels_i),
                       display_labels=C.NUTRISCORE_GRADES).plot(); plt.show()

# refit on ALL graded rows (encoded) and save; predict_grade() decodes 0..4 -> a..e
dfg = df.dropna(subset=[C.CLASSIFICATION_TARGET])
y_all = dfg[C.CLASSIFICATION_TARGET].map(g2i).values
best_clf.fit(dfg[C.NUTRIENT_FEATURES].values, y_all)
M.save_model(best_clf, M.CLF_PATH); print('saved classifier:', best_clf_name)


## 1.5 · Task 3 — Conditional VAE
Generate nutrient profiles conditioned on a target Nutri-Score grade.


In [ ]:
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import joblib

dfc = data.premium_subset(df)
X = dfc[C.NUTRIENT_FEATURES].astype(float).values
g = dfc[C.CLASSIFICATION_TARGET].map({x:i for i,x in enumerate(C.NUTRISCORE_GRADES)}).values
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
Coh = np.eye(len(C.NUTRISCORE_GRADES))[g]
ds = TensorDataset(torch.tensor(Xs, dtype=torch.float32), torch.tensor(Coh, dtype=torch.float32))
loader = DataLoader(ds, batch_size=256, shuffle=True)


In [ ]:
model = cvae.CVAE(); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(60):
    tot = 0.0
    for xb, cb in loader:
        recon, mu, logvar = model(xb, cb)
        loss, _, _ = cvae.loss_fn(recon, xb, mu, logvar, beta=1.0)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    if epoch % 15 == 0: print(f'epoch {epoch}  loss {tot/len(loader):.3f}')
torch.save(model.state_dict(), cvae.CVAE_PATH); joblib.dump(scaler, cvae.SCALER_PATH)
print('CVAE saved')


In [ ]:
# generate 3 healthy (grade-a) profiles and score them with the Phase-1 models
for prof in cvae.generate_profile('a', n=3):
    kcal = M.predict_calories(prof); grade = M.predict_grade(prof)
    print(f"~{kcal:.0f} kcal | predicted grade {grade} | " +
          ', '.join(f'{k.split(chr(95))[0]}={v:.1f}' for k, v in prof.items()))


---
# Phase 2 — RAG nutrition assistant


## 2.1 · Build the knowledge corpus (Wikipedia)
A compact, fully open corpus of nutrition/exercise topics. Extend the list as you like.


In [ ]:
import wikipediaapi
wiki = wikipediaapi.Wikipedia(user_agent='NutriCoach-capstone/0.1', language='en')
TITLES = ['Nutrition','Protein (nutrient)','Carbohydrate','Dietary fiber','Fat',
          'Saturated fat','Sugar','Vitamin','Mineral (nutrient)','Calorie',
          'Dietary supplement','Creatine','Whey protein','Glycemic index',
          'Nutri-Score','Healthy diet','Exercise physiology','Muscle hypertrophy',
          'Basal metabolic rate','Micronutrient']
documents = []
for t in TITLES:
    page = wiki.page(t)
    if page.exists() and len(page.text) > 200:
        documents.append({'text': page.text, 'source': f'Wikipedia: {t}'})
    else:
        print('skip', t)
chunks = ingest.build_corpus(documents)
print(len(documents), 'articles ->', len(chunks), 'chunks')


## 2.2 · Baseline retriever + index


In [ ]:
base = retriever.DenseRetriever(C.BASE_EMBEDDING_MODEL)
base.build_index([c['text'] for c in chunks], [c['meta'] for c in chunks])
base.search('Is whey protein useful after a workout?', k=3)


## 2.3 · Load the generator LLM
Small instruct model used both for synthetic-question generation and for RAG answers.


In [ ]:
from transformers import pipeline
llm = pipeline('text-generation', model=C.GENERATOR_MODEL,
               torch_dtype=torch.float16, device_map='auto')
def generate_fn(prompt, max_new_tokens=200):
    out = llm([{'role':'user','content':prompt}], max_new_tokens=max_new_tokens, do_sample=False)
    return out[0]['generated_text'][-1]['content']
print(generate_fn('Say OK if you can read this.', max_new_tokens=8))
llm.tokenizer.pad_token = llm.tokenizer.pad_token or llm.tokenizer.eos_token
llm.tokenizer.padding_side = 'left'


## 2.4 · Synthetic (query, passage) pairs
The LLM writes questions each chunk answers -> training pairs with no manual labels.
_(subset kept small so it finishes quickly; raise the cap for stronger fine-tuning.)_


In [ ]:
import json
from nutricoach import pairs

subset = chunks[:150]                      # enough for a clear signal
prompts = [[{'role':'user','content': pairs.QUESTION_PROMPT.format(n=2, passage=ch['text'])}]
           for ch in subset]
# batched generation — far faster than one call per chunk
outs = llm(prompts, max_new_tokens=64, do_sample=False, batch_size=16)

rows = []
for ch, out in zip(subset, outs):
    text = out[0]['generated_text'][-1]['content']
    qs = [q.strip(' -0123456789.') for q in text.splitlines() if len(q.strip()) > 8]
    for q in qs[:2]:
        rows.append({'query': q, 'positive': ch['text'], 'meta': ch.get('meta', {})})

with open(C.PAIRS_DIR / 'all_pairs.jsonl', 'w') as f:
    for r in rows:
        f.write(json.dumps(r) + '\n')
n_tr, n_te = pairs.split_pairs(test_frac=0.15)
print('pairs:', len(rows), '| train:', n_tr, '| test:', n_te)


## 2.5 · Fine-tune the retriever
MultipleNegativesRankingLoss (in-batch negatives).


In [ ]:
out_dir = retriever.finetune(epochs=1, batch_size=32)
print('fine-tuned retriever saved ->', out_dir)


## 2.6 · Evaluation — baseline vs fine-tuned (the headline result)


In [ ]:
tuned = retriever.DenseRetriever(str(C.FINETUNED_MODEL_DIR))
tuned.build_index([c['text'] for c in chunks], [c['meta'] for c in chunks])
results = E.compare_retrievers(base, tuned)
res_df = pd.DataFrame(results).T
print('Copy this table into the README:'); res_df


## 2.7 · Full RAG demo (retrieval + generation + Phase-1 tools)


In [ ]:
pipe = rag.RagPipeline(tuned, generate_fn)
out = pipe.answer('Why is protein important for muscle recovery?')
print(out['answer'])
print('\nSources:', [h['meta']['source'] for h in out['sources']])


In [ ]:
# quantitative question -> Phase-1 model tools
feat = {'proteins_100g':10,'fat_100g':0.4,'carbohydrates_100g':3.6,'sugars_100g':3.2}
print('calories:', round(pipe.tool_answer('predict_calories', features=feat)))
print('grade   :', pipe.tool_answer('predict_grade', features=feat))
print('generated grade-a profile:', pipe.tool_answer('generate_profile', target_grade='a', n=1))


---
## 3 · Persist to Hugging Face Hub
Runtime storage is ephemeral, so push the trained artifacts to the Hub. This uploads the whole
`models/` folder (fine-tuned retriever + tabular models + CVAE) into one repo. The repo id is
derived from your token — no username hardcoding.

Needs a Colab Secret 🔑 `HF_TOKEN` with **write** access (huggingface.co → Settings → Access Tokens).


In [ ]:
from huggingface_hub import login, whoami, HfApi, create_repo
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()   # falls back to an interactive token prompt

user = whoami()['name']
REPO = f'{user}/{C.HF_REPO_NAME}'
create_repo(REPO, repo_type='model', exist_ok=True)
HfApi().upload_folder(folder_path='models', repo_id=REPO, repo_type='model',
                      commit_message='Add NutriCoach trained artifacts')
print('pushed ->', f'https://huggingface.co/{REPO}')


### Reload from the Hub (in a fresh session, skip retraining)


In [ ]:
from huggingface_hub import snapshot_download
from sentence_transformers import SentenceTransformer
import joblib

local = snapshot_download(repo_id=REPO, repo_type='model')
retriever_model = SentenceTransformer(f'{local}/retriever_finetuned')
regressor  = joblib.load(f'{local}/regressor.joblib')
classifier = joblib.load(f'{local}/classifier.joblib')
print('reloaded artifacts from the Hub')
